# 00 — Iterators: The Foundation of Everything

**Goal:** Understand Python's iterator protocol deeply — because generators, coroutines, and async/await ALL build on this.

## The iterator protocol

Every `for` loop in Python does this under the hood:

```python
# for x in something:
#     ...
#
# is actually:
iterator = iter(something)    # calls something.__iter__()
while True:
    try:
        x = next(iterator)    # calls iterator.__next__()
    except StopIteration:
        break
```

Two methods. That's it. That's the whole protocol.

## Exercise 0.1 — See the protocol in action

Do the following:
1. Create a list `nums = [10, 20, 30]`
2. Get an iterator from it using `iter()`
3. Print the type of the list vs the type of the iterator
4. Call `next()` three times to pull values one by one
5. Call `next()` a fourth time — catch the `StopIteration` exception

Observe: a **list** is iterable (has `__iter__`) but is NOT an iterator itself.

In [9]:
# Exercise 0.1 — your code here
nums: list[int] = [10,20,30]
iterator = iter(nums) # wjhy iterator = nums.__iter__() works, but this line does not work ?
print('type of nums: ',type(nums))
print('type of iterator: ',type(iterator))
# print(iterator.__next__())
# print(iterator.__next__())
# print(iterator.__next__())
# print(iterator.__next__())
next(iterator)
next(iterator)




type of nums:  <class 'list'>
type of iterator:  <class 'list_iterator'>


20

TypeError: 'list' object is not an iterator

### Question 0.1
What is the difference between an **iterable** and an **iterator**? Can you call `next()` on a list directly?

*Your answer:*
We can get iterator from an iterable object (__iterator__() or iterator() ) - we can't call next() on iteratable object !

## Exercise 0.2 — Build your own `range()` from scratch

Python's `range(5)` doesn't create a list of 5 numbers. It creates an **iterable** that produces numbers on demand.

Build `MyRange` with TWO classes:
- `MyRange` — the iterable. Has `__init__(start, stop, step)` and `__iter__()` that returns a fresh iterator.
- `MyRangeIterator` — the iterator. Has `__init__`, `__iter__` (returns self), and `__next__` (returns next value or raises `StopIteration`).

Handle the single-arg case: `MyRange(5)` should mean `start=0, stop=5`.

Test with:
```python
for i in MyRange(5):          # 0 1 2 3 4
for i in MyRange(2, 10, 3):   # 2 5 8
```

In [ ]:
# Exercise 0.2 — Build MyRange + MyRangeIterator


### Question 0.2
Why does `MyRange` need TWO classes (iterable + iterator)? What would break if `MyRange` was also the iterator (i.e., had `__next__` directly)?

**Hint:** Try using the same `MyRange(3)` in two nested `for` loops:
```python
r = MyRange(3)
for i in r:
    for j in r:
        print(f"({i},{j})", end=" ")
```
Expected: `(0,0) (0,1) (0,2) (1,0) ...` — 9 pairs. Does it work if both loops share one iterator?

*Your answer:*


In [ ]:
# Prove it: nested loops over the same iterable


## Exercise 0.3 — Build your own `enumerate()` from scratch

Build `MyEnumerate(iterable, start=0)` that yields `(index, value)` tuples.

It should be both iterable AND iterator (single class is fine here — think about why).

Test:
```python
for i, val in MyEnumerate(['apple', 'banana', 'cherry']):
    print(f"{i}: {val}")  # 0: apple / 1: banana / 2: cherry
```

In [ ]:
# Exercise 0.3 — Build MyEnumerate


## Exercise 0.4 — The key insight: LAZY evaluation

Iterators don't compute everything upfront. They produce values ONE AT A TIME.

Use `sys.getsizeof()` to compare:
- `list(range(1_000_000))` — how many bytes?
- `range(1_000_000)` — how many bytes?

The lazy version uses almost NO memory — it just knows start, stop, step and computes each value when asked.

In [ ]:
import sys

# Exercise 0.4 — Compare eager vs lazy memory usage


### Question 0.4
Why is laziness important for concurrency? Think about the pattern:
- An iterator produces a value → pauses → produces next value
- A coroutine runs → awaits → resumes → awaits
- Same pattern: **produce → pause → resume → produce**

The iterator protocol (`__next__` → value → `__next__` → value → StopIteration) is the heartbeat of concurrency.

*Your answer:*


## Exercise 0.5 — Build your own `zip()` from scratch

Build `MyZip(*iterables)` that iterates MULTIPLE iterables in lockstep.

Key: in `__init__`, convert each iterable to an iterator using `iter()`. In `__next__`, call `next()` on ALL iterators and return a tuple. If ANY raises `StopIteration`, you stop.

Test:
```python
for a, b in MyZip([1,2,3], ['x','y','z']):
    print(a, b)   # 1 x / 2 y / 3 z

for a, b, c in MyZip('abc', [1,2,3], [True, False, True]):
    print(a, b, c)
```

In [ ]:
# Exercise 0.5 — Build MyZip


## Summary

| Concept | What it means |
|---------|---------------|
| **Iterable** | Has `__iter__()` — can give you an iterator |
| **Iterator** | Has `__next__()` — produces values one at a time |
| **StopIteration** | Signal that the iterator is exhausted |
| **Lazy evaluation** | Don't compute until asked — same pattern as coroutines |
| **`for` loop** | Just `iter()` + `next()` + catch `StopIteration` |

**Key insight for later:** Replace `__next__` returning a value with `yield` returning a value, and you get a **generator**. Replace `yield` with `await`, and you get a **coroutine**. Same pattern, different syntax.

**Next notebook:** Generators — iterators without the boilerplate